In [17]:
!pip install -q gymnasium numpy matplotlib


In [23]:
# =========================================================
# Paper Topic: Adaptive Skill Coordination for Robotic Mobile Manipulation
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class VisualEncoder(nn.Module):
    def __init__(self, input_channels=3, output_dim=512):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten()
        )
        dummy = torch.zeros(1, input_channels, 84, 84)
        self.flat_size = self.cnn(dummy).shape[1]
        self.fc = nn.Linear(self.flat_size, output_dim)

    def forward(self, x):
        x = self.cnn(x)
        return F.relu(self.fc(x))

class VisuomotorPolicy(nn.Module):
    def __init__(self, visual_dim, state_dim, action_dim, hidden_dim=512):
        super().__init__()
        input_dim = visual_dim + state_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        self.actor_mean = nn.Linear(hidden_dim, action_dim)
        self.actor_logstd = nn.Parameter(torch.zeros(1, action_dim))
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, visual_obs, state_obs, hidden_state):
        if visual_obs.shape[-1] > 0:
            x = torch.cat([visual_obs, state_obs], dim=-1)
        else:
            x = state_obs

        x = self.mlp(x)
        hidden_state = self.gru(x, hidden_state)

        mean = torch.tanh(self.actor_mean(hidden_state))
        std = torch.exp(self.actor_logstd).clamp(1e-6, 1.0)
        value = self.critic(hidden_state)

        return mean, std, value, hidden_state

class ASC_Agent(nn.Module):
    def __init__(self):
        super().__init__()
        self.vis_enc_nav = VisualEncoder(input_channels=3, output_dim=512)
        self.vis_enc_pick = VisualEncoder(input_channels=3, output_dim=512)

        self.nav_skill = VisuomotorPolicy(visual_dim=512, state_dim=3, action_dim=2)
        self.pick_skill = VisuomotorPolicy(visual_dim=512, state_dim=4, action_dim=4)
        self.place_skill = VisuomotorPolicy(visual_dim=0, state_dim=7, action_dim=4)

        self.gating_net = nn.Sequential(
            nn.Linear(512 + 512 + 10, 256),
            nn.ReLU(),
            nn.Linear(256, 3)
        )

        self.corrective_net = VisuomotorPolicy(visual_dim=1024, state_dim=10, action_dim=6)

    def forward(self, obs, hidden_states):
        emb_nav = self.vis_enc_nav(obs['I_front'])
        emb_pick = self.vis_enc_pick(obs['I_grip'])

        gate_input = torch.cat([emb_nav, emb_pick, obs['state_coord']], dim=-1)
        gate_logits = self.gating_net(gate_input)

        gate_probs = torch.softmax(gate_logits, dim=-1)
        g = (gate_probs > 0.5).float()

        arm_idx = torch.argmax(gate_logits[:, 1:], dim=-1)
        g_pick = F.one_hot(arm_idx, num_classes=2)[:, 0].float()
        g_place = F.one_hot(arm_idx, num_classes=2)[:, 1].float()
        g_nav = g[:, 0]

        a_nav, _, _, hidden_states['nav'] = self.nav_skill(emb_nav, obs['state_nav'], hidden_states['nav'])
        a_pick, _, _, hidden_states['pick'] = self.pick_skill(emb_pick, obs['state_pick'], hidden_states['pick'])

        # FIX: Pass an empty tensor for visual input since visual_dim=0 for place_skill
        batch_size = obs['state_place'].shape[0]
        device = obs['state_place'].device
        a_place, _, _, hidden_states['place'] = self.place_skill(torch.empty(batch_size, 0, device=device), obs['state_place'], hidden_states['place'])

        a_base_MoE = g_nav.unsqueeze(-1) * a_nav
        a_arm_MoE = g_pick.unsqueeze(-1) * a_pick + g_place.unsqueeze(-1) * a_place

        corr_input = torch.cat([emb_nav, emb_pick], dim=-1)
        a_corr, _, _, hidden_states['corr'] = self.corrective_net(corr_input, obs['state_coord'], hidden_states['corr'])
        a_corr_base = a_corr[:, :2]
        a_corr_arm = a_corr[:, 2:]

        a_adapt_base = (1 - g_nav).unsqueeze(-1) * a_corr_base
        arm_active = (g_pick + g_place).unsqueeze(-1).clamp(max=1.0)
        a_adapt_arm = (1 - arm_active) * a_corr_arm

        a_base_ASC = a_base_MoE + a_adapt_base
        a_arm_ASC = a_arm_MoE + a_adapt_arm

        a_ASC = torch.cat([a_base_ASC, a_arm_ASC], dim=-1)

        return a_ASC, gate_probs, hidden_states

# --- EXECUTION DEMO ---
agent = ASC_Agent().to(device)

batch_size = 1
obs = {
    'I_front': torch.randn(batch_size, 3, 84, 84).to(device),
    'I_grip': torch.randn(batch_size, 3, 84, 84).to(device),
    'state_nav': torch.randn(batch_size, 3).to(device),
    'state_pick': torch.randn(batch_size, 4).to(device),
    'state_place': torch.randn(batch_size, 7).to(device),
    'state_coord': torch.randn(batch_size, 10).to(device)
}

hidden_states = {
    'nav': torch.zeros(batch_size, 512).to(device),
    'pick': torch.zeros(batch_size, 512).to(device),
    'place': torch.zeros(batch_size, 512).to(device),
    'corr': torch.zeros(batch_size, 512).to(device)
}

with torch.no_grad():
    action_ASC, gate_probs, hidden_states = agent(obs, hidden_states)

print("--- ASC Forward Pass Output ---")
print(f"Final Action (Base + Arm): {action_ASC.shape} -> {action_ASC.cpu().numpy()}")
print(f"Gating Probabilities (Nav, Pick, Place): {gate_probs.cpu().numpy()}")
print(f"Active Skill Interpretation: Nav={gate_probs[0,0].item():.2f}, Pick={gate_probs[0,1].item():.2f}, Place={gate_probs[0,2].item():.2f}")

Using device: cpu
--- ASC Forward Pass Output ---
Final Action (Base + Arm): torch.Size([1, 6]) -> [[-0.05377743  0.03267412 -0.06318937 -0.04269115 -0.03648308 -0.03680489]]
Gating Probabilities (Nav, Pick, Place): [[0.31449583 0.33086962 0.3546346 ]]
Active Skill Interpretation: Nav=0.31, Pick=0.33, Place=0.35
